In [ ]:
#Diagnostic vp004 artifact type

In [1]:
"""
Diagnostic: what specifically is wrong with VP004's EEG signal quality?

This script compares VP004 against 2 "healthy" reference subjects (VP001, VP002 --
subjects with consistently full window counts across all filter tests) on four
independent axes, each pointing to a DIFFERENT possible root cause:

  1. Bad-channel count (same detection logic as preprocess_eeg()) -- electrode/contact issues
  2. Overall signal variability (RMS) -- general noisiness
  3. Correlation of large-amplitude events with VEOG -- eye-blink hypothesis
  4. Spatial pattern of a few large-amplitude events (which channels are most affected) --
     frontal-only pattern = eye blink; broadband/all-channel pattern = muscle/motion artifact

Run this and compare VP004's numbers against the two reference subjects to see which
axis stands out.
"""
import os
import numpy as np
import scipy.io as sio
import mne
import matplotlib.pyplot as plt

BASE_DATASET_PATH = r"E:\kntu\Thesis\codes\DATA"
TEST_SUBJECT = "VP004"
REFERENCE_SUBJECTS = ["VP001", "VP002"]  # subjects with consistently full window counts


def load_eeg_raw(subject_id):
    eeg_folder = os.path.join(BASE_DATASET_PATH, f"{subject_id}-EEG")
    cnt_path = os.path.join(eeg_folder, "cnt_nback.mat")
    mat_cnt = sio.loadmat(cnt_path, struct_as_record=False, squeeze_me=True)
    cnt_struct = mat_cnt[[k for k in mat_cnt.keys() if not k.startswith('__')][0]]
    data = cnt_struct.x.T
    sfreq = float(cnt_struct.fs)
    ch_names = [str(ch).strip() for ch in cnt_struct.clab]

    if np.max(np.abs(data)) > 1.0:
        data = data * 1e-6
    if sfreq != 200.0:
        from scipy.signal import resample
        n_samples = int(data.shape[1] * 200.0 / sfreq)
        data = resample(data, n_samples, axis=1)
        sfreq = 200.0

    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types='eeg')
    raw = mne.io.RawArray(data.copy(), info, verbose=False)
    eog_channels = [ch for ch in ['VEOG', 'HEOG'] if ch in ch_names]
    if eog_channels:
        raw.set_channel_types({ch: 'eog' for ch in eog_channels})
    raw.set_montage('standard_1005', match_case=False, on_missing='ignore')
    return raw


def detect_bad_channels(raw):
    """Same bad-channel detection logic used in preprocess_eeg()."""
    raw_temp = raw.copy().filter(l_freq=10.0, h_freq=40.0, method='fir', verbose=False)
    eeg_picks = mne.pick_types(raw_temp.info, eeg=True, eog=False)
    eeg_names = [raw_temp.ch_names[idx] for idx in eeg_picks]

    temp_data = raw_temp.get_data(picks=eeg_picks)
    stds = np.std(temp_data, axis=1)
    median_std = np.median(stds)
    mad = np.median(np.abs(stds - median_std)) if np.median(np.abs(stds - median_std)) > 0 else 1e-6
    corr_matrix = np.corrcoef(temp_data)
    mean_corrs = np.mean(corr_matrix, axis=1)

    bad_channels = []
    for i, ch_name in enumerate(eeg_names):
        if ch_name.upper() in ['VEOG', 'HEOG', 'TP9']:
            continue
        if stds[i] > (median_std + 4.5 * mad) or stds[i] < (median_std - 4.5 * mad) or mean_corrs[i] < 0.30:
            bad_channels.append(ch_name)
    return bad_channels, dict(zip(eeg_names, stds)), dict(zip(eeg_names, mean_corrs))


print("=" * 90)
print(f"COMPARING {TEST_SUBJECT} AGAINST REFERENCE SUBJECTS {REFERENCE_SUBJECTS}")
print("=" * 90)

all_subjects = [TEST_SUBJECT] + REFERENCE_SUBJECTS
results = {}

for subject in all_subjects:
    print(f"\n--- {subject} ---")
    raw = load_eeg_raw(subject)

    # ----- CHECK 1: bad channel count -----
    bad_channels, stds, corrs = detect_bad_channels(raw)
    print(f"[Check 1] Bad channels detected: {len(bad_channels)} -> {bad_channels}")

    # ----- CHECK 2: overall RMS (broadband, pre-filter) -----
    data_full = raw.get_data(picks=mne.pick_types(raw.info, eeg=True, eog=False))
    overall_rms = np.sqrt(np.mean(data_full ** 2)) * 1e6  # in uV
    print(f"[Check 2] Overall broadband RMS: {overall_rms:.2f} uV")

    # ----- CHECK 3: correlation of large-amplitude events with VEOG -----
    veog_corr = None
    if 'VEOG' in raw.ch_names:
        veog_data = raw.get_data(picks=['VEOG'])[0]
        eeg_picks_names = [ch for ch in raw.ch_names if ch.upper() not in ['VEOG', 'HEOG']]
        eeg_data_only = raw.get_data(picks=eeg_picks_names)
        global_envelope = np.max(np.abs(eeg_data_only), axis=0)  # per-timepoint max across channels
        veog_corr = np.corrcoef(global_envelope, np.abs(veog_data))[0, 1]
        print(f"[Check 3] Correlation of large-amplitude events with |VEOG|: {veog_corr:.3f}")
        print("          (closer to 1 = large events are mostly eye blinks)")
    else:
        print("[Check 3] No VEOG channel found -- skipped")

    # ----- CHECK 4: spatial pattern of the single largest-amplitude event -----
    eeg_picks_names = [ch for ch in raw.ch_names if ch.upper() not in ['VEOG', 'HEOG']]
    eeg_data_only = raw.get_data(picks=eeg_picks_names)
    global_envelope = np.max(np.abs(eeg_data_only), axis=0)
    peak_idx = np.argmax(global_envelope)
    peak_window = eeg_data_only[:, max(0, peak_idx - 20):peak_idx + 20]
    per_channel_peak = np.max(np.abs(peak_window), axis=1) * 1e6  # uV per channel at this event

    frontal_channels = [i for i, ch in enumerate(eeg_picks_names) if ch.upper().startswith(('FP', 'AF', 'F'))]
    other_channels = [i for i in range(len(eeg_picks_names)) if i not in frontal_channels]
    frontal_peak = np.mean(per_channel_peak[frontal_channels]) if frontal_channels else np.nan
    other_peak = np.mean(per_channel_peak[other_channels]) if other_channels else np.nan
    print(f"[Check 4] Largest event -- mean peak amplitude: frontal channels={frontal_peak:.1f}uV, "
          f"other channels={other_peak:.1f}uV, ratio={frontal_peak/other_peak if other_peak>0 else float('nan'):.2f}")
    print("          (ratio >> 1 = localized to frontal, likely eye-related;")
    print("           ratio ~ 1 = broadband/global, likely muscle or motion artifact)")

    results[subject] = {
        'n_bad_channels': len(bad_channels),
        'bad_channels': bad_channels,
        'overall_rms_uv': overall_rms,
        'veog_corr': veog_corr,
        'frontal_other_ratio': frontal_peak / other_peak if other_peak and other_peak > 0 else None,
    }

# ----- SUMMARY TABLE -----
print("\n" + "=" * 90)
print("SUMMARY TABLE")
print("=" * 90)
print(f"{'Subject':<10}{'#Bad Ch':>10}{'RMS(uV)':>12}{'VEOG corr':>12}{'Frontal/Other':>16}")
for subject in all_subjects:
    r = results[subject]
    veog_str = f"{r['veog_corr']:.3f}" if r['veog_corr'] is not None else "N/A"
    ratio_str = f"{r['frontal_other_ratio']:.2f}" if r['frontal_other_ratio'] is not None else "N/A"
    print(f"{subject:<10}{r['n_bad_channels']:>10}{r['overall_rms_uv']:>12.2f}{veog_str:>12}{ratio_str:>16}")

print("\n" + "=" * 90)
print("INTERPRETATION GUIDE:")
print("- If VP004 has notably MORE bad channels than references -> electrode/contact issue")
print("- If VP004's RMS is much higher -> general noisiness (could be many causes)")
print("- If VP004's VEOG correlation is high (and higher than references) -> eye blinks dominate")
print("- If VP004's frontal/other ratio is >>1 -> localized frontal (eye) artifact")
print("- If VP004's frontal/other ratio is ~1 (close to references) -> broadband/muscle/motion")
print("  artifact, not eye-related -- this would explain why ICA (which targets eye components")
print("  specifically) doesn't fully clean this subject's data.")
print("=" * 90)

COMPARING VP004 AGAINST REFERENCE SUBJECTS ['VP001', 'VP002']

--- VP004 ---
[Check 1] Bad channels detected: 3 -> ['T7', 'CP5', 'P7']
[Check 2] Overall broadband RMS: 330.47 uV
[Check 3] Correlation of large-amplitude events with |VEOG|: 0.599
          (closer to 1 = large events are mostly eye blinks)
[Check 4] Largest event -- mean peak amplitude: frontal channels=993.2uV, other channels=414.3uV, ratio=2.40
          (ratio >> 1 = localized to frontal, likely eye-related;
           ratio ~ 1 = broadband/global, likely muscle or motion artifact)

--- VP001 ---
[Check 1] Bad channels detected: 3 -> ['Fp1', 'T7', 'Fp2']
[Check 2] Overall broadband RMS: 173.70 uV
[Check 3] Correlation of large-amplitude events with |VEOG|: 0.588
          (closer to 1 = large events are mostly eye blinks)
[Check 4] Largest event -- mean peak amplitude: frontal channels=483.2uV, other channels=89.2uV, ratio=5.42
          (ratio >> 1 = localized to frontal, likely eye-related;
           ratio ~ 1 = br